# Audio Event Detection — Training Notebook (Colab)

This notebook trains the multi-label audio event detection model using data **already saved to Google Drive**.

**Prerequisites:** Run the **data setup notebook** (`colab_data_setup.ipynb`) first to download and prepare the data.

Training is **resumable** across Colab sessions — checkpoints are saved to Google Drive automatically.

## 1. Setup & Mount Drive

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
drive.mount('/content/drive')

# Clone or update the project
import os
PROJECT_DIR = '/content/audio-event-detection'
DRIVE_DIR = '/content/drive/MyDrive/audio-event-detection'

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'checkpoints'), exist_ok=True)
os.makedirs(os.path.join(DRIVE_DIR, 'logs'), exist_ok=True)

In [ ]:
# Install dependencies
!pip install -q torch torchaudio pytorch-lightning timm librosa soundfile scikit-learn pyyaml matplotlib tqdm

In [ ]:
import subprocess

if os.path.exists(PROJECT_DIR):
    print("Repo already exists, pulling latest...")
    subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/ankur-paul/audio-event-detection.git", PROJECT_DIR], check=True)

os.chdir(PROJECT_DIR)

In [ ]:
!curl -L -o spectrograms.zip https://www.kaggle.com/api/v1/datasets/download/ankurpaul52/spectrograms
!unzip -q spectrograms.zip -d /content/audio-event-detection/data/


In [ ]:
import shutil

local_data = os.path.join(PROJECT_DIR, 'data')
drive_data = os.path.join(DRIVE_DIR, 'data')

# Verify key files exist
assert os.path.exists('data/labels.csv'), "labels.csv not found — run data setup notebook first"
assert os.path.exists('data/class_map.json'), "class_map.json not found — run data setup notebook first"
print("✓ Data files verified")

## 2. Configuration

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from src.utils.config import load_config

# Load config with Drive checkpoint dir for persistence
config = load_config(overrides={
    'training': {
        'epochs': 50,
        'batch_size': 32,
        'num_workers': 2,
    },
})

print(f'Config loaded. Device: {"cuda" if __import__("torch").cuda.is_available() else "cpu"}')

## 3. Prepare Data

In [ ]:
from src.utils.logger import setup_logger
from src.data.dataset_preparation import (
    create_class_map, load_class_map, parse_labels_csv, split_dataset
)
from src.data.dataset import create_dataloaders

logger = setup_logger(log_level='INFO', console=True)

# Create class map
class_map = create_class_map(save_path=config.paths.class_map_file)
config.model.num_classes = len(class_map)
print(f'{len(class_map)} sound classes')

# Load labels and split
entries = parse_labels_csv(config.paths.labels_csv)
train_entries, val_entries, test_entries = split_dataset(entries)

# Create data loaders
train_loader, val_loader = create_dataloaders(
    train_entries, val_entries, class_map, config
)
print(f'Train: {len(train_entries)}, Val: {len(val_entries)}, Test: {len(test_entries)}')

## 4. Build Model

In [ ]:
from src.models.audio_event_model import build_model

model = build_model(config)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')

## 5. Train (Resumable)

This cell will automatically resume from the latest checkpoint if one exists.
Checkpoints are saved to both local storage and Google Drive.

In [ ]:
import glob
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, LearningRateMonitor
from src.training.trainer import (
    AudioEventLightningModule,
    MetricsLoggerCallback,
    DriveCheckpointCallback,
)

# Wrap model in Lightning module
lit_module = AudioEventLightningModule(model=model, config=config)

# Callbacks
ckpt_cfg = config.checkpoint
callbacks = [
    ModelCheckpoint(
        dirpath=config.paths.checkpoint_dir,
        filename="epoch{epoch:04d}-val_mAP{val_mAP:.4f}",
        auto_insert_metric_name=False,
        monitor=f"val_{ckpt_cfg.best_metric}",
        mode="max",
        save_top_k=ckpt_cfg.keep_last_n,
        save_last=True,
        every_n_epochs=ckpt_cfg.save_every_n_epochs,
    ),
    MetricsLoggerCallback(metrics_file=config.paths.metrics_file),
    LearningRateMonitor(logging_interval="epoch"),
    DriveCheckpointCallback(
        drive_checkpoint_dir=config.paths.drive_checkpoint_dir,
    ),
]

# Auto-resume from latest checkpoint
ckpt_files = sorted(glob.glob(os.path.join(config.paths.checkpoint_dir, "*.ckpt")),
                     key=os.path.getmtime)
resume_ckpt = ckpt_files[-1] if ckpt_files else None
if resume_ckpt:
    print(f"Resuming from: {resume_ckpt}")

# Train
trainer = pl.Trainer(
    max_epochs=config.training.epochs,
    accelerator="gpu" if __import__("torch").cuda.is_available() else "cpu",
    devices="auto",
    precision="16-mixed" if config.training.mixed_precision else 32,
    gradient_clip_val=getattr(config.training, "gradient_clip_norm", None),
    callbacks=callbacks,
    default_root_dir=config.paths.log_dir,
    log_every_n_steps=50,
)

trainer.fit(
    model=lit_module,
    train_dataloaders=train_loader,
    val_dataloaders=val_loader,
    ckpt_path=resume_ckpt,
)

## 6. Visualize Results

In [ ]:
from src.training.experiment_tracker import MetricsTracker
from src.visualization.visualizer import plot_training_curves

tracker = MetricsTracker(metrics_file=config.paths.metrics_file)
history = tracker.load_history()

fig = plot_training_curves(history)
display(fig)

## 7. Inference

In [ ]:
from src.inference.inference_pipeline import InferencePipeline
from src.training.checkpoint import find_best_checkpoint, load_model_from_checkpoint
from src.visualization.visualizer import plot_event_timeline

# Load best model
ckpt_path = find_best_checkpoint(
    checkpoint_dir=config.paths.checkpoint_dir,
    drive_dir=config.paths.drive_checkpoint_dir,
)
if ckpt_path:
    load_model_from_checkpoint(model=model, checkpoint_path=ckpt_path)
else:
    print("No checkpoint found.")

# Create inference pipeline
pipeline = InferencePipeline(model=model, class_map=class_map, config=config)

# Run on a test file
# result = pipeline.predict_file('path/to/test_audio.wav')
# fig = plot_event_timeline(result)
# display(fig)